# Web Scraping — KaBuM!

## Objetivo

Extrair **nome** e **preço** dos produtos de **Memória RAM** do site KaBuM.

## Imports

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import random
import pathlib
import re
import pandas as pd

## Configurações

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9",
}

## Abordagem 1 — requests + BeautifulSoup

> ⚠️ O KaBuM usa React/Next.js. Se o `<main>` não for encontrado no HTML estático, use a **Abordagem 2**.

In [ ]:
url = "https://www.kabum.com.br/hardware/memoria-ram"

response = requests.get(url, headers=headers)
print("Status:", response.status_code)

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
# Diagnóstico: verificar o que o requests realmente recebeu
print("Existe <main>?", soup.find("main") is not None)
print("Existe <body>?", soup.find("body") is not None)
print("\n--- Início do HTML ---")
print(response.text[:2000])

In [ ]:
# Localiza o <main> e coleta todos os <a href> dentro dele
main = soup.find("main")

if main:
    print("✅ <main> encontrado")
    links_produtos = main.find_all("a", href=True)
else:
    print("⚠️  <main> não encontrado — tentando fallback em todo o HTML")
    links_produtos = [
        a for a in soup.find_all("a", href=True)
        if "/produto/" in a.get("href", "")
    ]

print(f"Links de produto encontrados: {len(links_produtos)}")

if len(links_produtos) == 0:
    print("\n❌ Nenhum produto encontrado no HTML estático. Use a Abordagem 2.")
else:
    # Inspeciona o primeiro para verificar a estrutura
    print("\nPrimeiro link encontrado:")
    print(links_produtos[0])

In [ ]:
def extrair_preco(texto):
    match = re.search(r'R\$\s?[\d\.]+,[\d]+', texto)
    return match.group(0) if match else None

produtos_bs4 = []

for tag in links_produtos:
    href = tag.get("href", "")
    label = tag.get("aria-label", "")
    preco = extrair_preco(label)

    if "/produto/" in href and preco:
        produtos_bs4.append({
            "nome": label.split(",")[0].strip(),
            "preco": preco,
            "url": "https://www.kabum.com.br" + href if href.startswith("/") else href,
        })

print(f"Produtos extraídos: {len(produtos_bs4)}")
produtos_bs4[:3]

---
## Abordagem 2 — API interna do KaBuM (recomendada) ✅

Retorna JSON estruturado diretamente, sem depender de JavaScript.

**Parâmetros:**
- `category`: 2560 = Memória RAM
- `page_number`: página (começa em 1)
- `page_size`: produtos por página (máx. recomendado: 20)

In [3]:
API_URL = "https://servicespub.prod.api.aws.grupokabum.com.br/catalog/v2/products"

headers_api = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Origin": "https://www.kabum.com.br",
    "Referer": "https://www.kabum.com.br/",
}

params = {
    "page_number": 1,
    "page_size": 20,
    "aplicacao": "loja",
    "category": 2560,
}

response_api = requests.get(API_URL, params=params, headers=headers_api)
print("Status:", response_api.status_code)

Status: 200


In [4]:
# Inspeciona a estrutura real da resposta
import pprint

data = response_api.json()
print("Chaves na raiz da resposta:", list(data.keys()))
print("\nChaves de um produto bruto:")
pprint.pprint(list(data["data"][0].keys()))
print("\nConteúdo completo do primeiro produto:")
pprint.pprint(data["data"][0])

Chaves na raiz da resposta: ['meta', 'links', 'data']

Chaves de um produto bruto:
['type', 'id', 'links', 'relationships', 'attributes']

Conteúdo completo do primeiro produto:
{'attributes': {'age_rating': 0,
                'available': True,
                'date_pre_order': 0,
                'description': 'Características:<br /> - Marca: Kross<br /> - '
                               'Modelo: KE-CBS0112<br /> Especificações:<br /> '
                               '- Comprimento do Cabo: 10 CM<br /> - Função: '
                               'Transforma a saída molex de sua fonte em '
                               'Sata<br /> Conteúdo da embalagem:<br /> - 1 x '
                               'Cabo',
                'discount_percentage': 0,
                'external_url': '',
                'featured_product': False,
                'has_free_shipping': False,
                'has_free_shipping_for_prime_user': False,
                'images': ['https://images.kabum.com.br/pro

In [6]:
# Estrutura real da API:
#   produto['id']              -> id do produto
#   produto['attributes'][...] -> todos os demais campos
def extrair_produto(produto):
    attr = produto.get('attributes', {})
    return {
        'id':             produto.get('id'),
        'nome':           attr.get('title'),
        'preco_original': attr.get('old_price'),
        'preco_atual':    attr.get('price'),
        'preco_pix':      attr.get('price_with_discount'),
        'desconto_pct':   attr.get('discount_percentage'),
        'avaliacao':      attr.get('score_of_ratings'),
        'num_avaliacoes': attr.get('number_of_ratings'),
        'estoque':        attr.get('stock'),
        'fabricante':     attr.get('manufacturer', {}).get('name'),
        'url': f"https://www.kabum.com.br/produto/{produto.get('id')}/{attr.get('product_link', '')}",
    }

produtos_pagina = [extrair_produto(p) for p in data['data']]
print(f'Produtos na página 1: {len(produtos_pagina)}')
produtos_pagina[0]

Produtos na página 1: 20


{'id': 643989,
 'nome': 'Cabo De Forca Sata Kross 10cm Ke-cbs0112',
 'preco_original': 0.0,
 'preco_atual': 8.9,
 'preco_pix': 8.9,
 'desconto_pct': 0,
 'avaliacao': 0,
 'num_avaliacoes': 0,
 'estoque': 3,
 'fabricante': 'Kross',
 'url': 'https://www.kabum.com.br/produto/643989/cabo-de-forca-sata-kross-10cm-ke-cbs0112'}

## Coletando múltiplas páginas

In [7]:
PAGE_SIZE = 20
NUM_PAGINAS = 5  # Aumente para coletar mais

todos_produtos = []

for pagina in range(1, NUM_PAGINAS + 1):
    p = {"page_number": pagina, "page_size": PAGE_SIZE, "aplicacao": "loja", "category": 2560}
    resp = requests.get(API_URL, params=p, headers=headers_api)
    print(f"Página {pagina}: {resp.status_code}")

    if resp.status_code != 200:
        print("Erro, pulando página.")
        continue

    todos_produtos.extend([extrair_produto(x) for x in resp.json().get("data", [])])
    time.sleep(random.uniform(0.5, 1.5))

print(f"\nTotal coletado: {len(todos_produtos)}")

Página 1: 200
Página 2: 200
Página 3: 200
Página 4: 200
Página 5: 200

Total coletado: 100


## Salvando os dados

In [8]:
data_dir = pathlib.Path("data")
data_dir.mkdir(exist_ok=True)

# JSON
with (data_dir / "kabum_memoria_ram.json").open("w", encoding="utf-8") as f:
    json.dump(todos_produtos, f, ensure_ascii=False, indent=4)

# CSV
df = pd.DataFrame(todos_produtos)
df.to_csv(data_dir / "kabum_memoria_ram.csv", index=False, encoding="utf-8")

print("Arquivos salvos em:", data_dir)
df.head()

Arquivos salvos em: data


,id,nome,preco_original,preco_atual,preco_pix,desconto_pct,avaliacao,num_avaliacoes,estoque,fabricante,url
0,643989,Cabo De Forca Sata Kross 10cm Ke-cbs0112,0.0,8.90,8.90,0,0,0,3,Kross,https://www.kabum.com.br/produto/643989/cabo-d...
1,707598,Conector Dc Jack Para Notebook Hp Compaq Presa...,0.0,25.68,25.68,0,0,0,10,BRINGIT,https://www.kabum.com.br/produto/707598/None
2,561331,Bateria Para Notebook Dell Part Number 1w2y2,0.0,229.00,217.55,5,0,0,1,BRINGIT,https://www.kabum.com.br/produto/561331/None
3,631250,Bateria Para Notebook Asus K52je | 4000 Mah,0.0,139.00,132.05,5,0,0,5,BRINGIT,https://www.kabum.com.br/produto/631250/None
4,676671,Carregador De Tomada Plug Adaptador Fonte Usb ...,0.0,23.90,23.90,0,0,0,1,H'Mastonn,https://www.kabum.com.br/produto/676671/carreg...


## Análise rápida dos preços

In [ ]:
import matplotlib.pyplot as plt

print(df[["preco_atual", "preco_pix", "desconto_pct"]].describe())

plt.figure(figsize=(10, 5))
df["preco_atual"].dropna().hist(bins=30, color="steelblue", edgecolor="white")
plt.xlabel("Preço (R$)")
plt.ylabel("Quantidade")
plt.title("Distribuição de Preços — Memória RAM (KaBuM)")
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 mais baratos
df.nsmallest(10, "preco_atual")[["nome", "preco_atual", "preco_pix", "desconto_pct"]]

In [ ]:
# Top 10 maiores descontos
df.nlargest(10, "desconto_pct")[["nome", "preco_original", "preco_atual", "desconto_pct"]]